## DTCC IFW 21208 Withdrawal-Quote XML Generator

**End-to-end pipeline:** POV file → Flattened CSV → Field Selection → Validation → ACORD 21208 XML

| Step | Description |
|------|-------------|
| **1. Flatten POV** | Parse a DTCC POV fixed-width file into a wide CSV |
| **2. Explore Output** | Inspect the flattened data — contracts, agents, owners |
| **3. Map to XML Schema** | Select & map POV fields into the 21208 WD-Quote format |
| **3a. Sync Arrangement** | Sync `ModalAmt` and `ArrSubType` with withdrawal classification |
| **3b. Validation CSV** | Build pre-generation validation file; create timestamped run folder |
| **3c. Failures Analysis** | Audit mandatory fields; predict carrier-rejection error codes |
| **4. Generate XMLs** | Produce SOAP-wrapped ACORD XMLs + validation scorecard |
| **5. Evaluate Responses** | Load carrier results XLSX; cross-ref with validation; error breakdown |

In [0]:
import sys, os, importlib

PROJECT_ROOT = "/Workspace/Users/edward.m.ruiz@brighthousefinancial.com/XMLGenerator"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Clear cached modules so we always pick up latest code
for mod_name in list(sys.modules):
    if mod_name.startswith(("config.", "modules.", "main")):
        del sys.modules[mod_name]

from main import run_pipeline, analyze_external, flatten_pov
import main as _main_mod
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType
from config.schema_config import get_spark_schema, get_column_names, FIELD_DEFINITIONS
from config.pov_record_layouts import RECORD_LAYOUTS, get_field_names

# Serverless-compatible CSV loader override
def _load_csv_serverless(spark, csv_path, table_name, schema=None, database=None):
    if schema is None:
        schema = get_spark_schema()
    pdf = pd.read_csv(csv_path, dtype=str, encoding='ISO-8859-1')
    pdf = pdf.where(pdf.notna(), None)
    nullable_schema = StructType([StructField(f.name, f.dataType, True) for f in schema.fields])
    df = spark.createDataFrame(pdf, schema=nullable_schema)
    df.createOrReplaceTempView(table_name)
    return df

_main_mod.load_csv = _load_csv_serverless

BASE_DIR = "/Workspace/Users/edward.m.ruiz@brighthousefinancial.com/XMLGenerator/run_output"
print("Setup complete – all modules loaded.")

### Step 1 — Flatten POV File
Parse a DTCC POV fixed-width file into a flat CSV. Each contract becomes one wide row
with columns like `1301_Contract_Number`, `1305_Agent_Last_Name`, `1309_Party_First_Name`, etc.

> **To use your own file:** change `POV_INPUT_PATH` below to point to your real POV file.

In [0]:
from config.pov_record_layouts import RECORD_LAYOUTS as _RL

_POV_DIR = os.path.join(BASE_DIR, "data", "POV")
os.makedirs(_POV_DIR, exist_ok=True)

# ╔══════════════════════════════════════════════════════════════════╗
# ║  POV file to flatten — update filename as needed                ║
# ║  Drop files into: run_output/data/POV/                         ║
# ╚══════════════════════════════════════════════════════════════════╝
POV_INPUT_PATH = os.path.join(_POV_DIR, "E9914T.BHF.DXCA.DTCC.POV.P.DATA.20260320_0305.txt")

print(f"POV file: {POV_INPUT_PATH}")
print(f"Exists:   {os.path.isfile(POV_INPUT_PATH)}")

pov_summary = flatten_pov(
    input_path=POV_INPUT_PATH,
    base_dir=os.path.join(BASE_DIR, "output", "pov"),
)

print(f"\nFile format   : {pov_summary.get('file_format', 'N/A')}")
print(f"Flattened CSV : {pov_summary['csv_path']}")
print(f"Contracts     : {pov_summary['contract_count']}")
print(f"Validation    : {'PASS' if pov_summary['valid'] else 'FAIL'}")

### Step 2 — Explore Flattened Output
Inspect the flattened CSV to see what fields are available. The key columns
for the XML generator are:

| POV Field | Description |
|-----------|-------------|
| `1301_Contract_Number` | Policy number |
| `1301_CUSIP_Number` | CUSIP identifier |
| `1301_Distributors_Account_ID` | Distributor account |
| `1305_Agent_First/Last_Name` | Agent identity |
| `1305_National_Producer_Number` | Agent NIPR |
| `1309_Party_First/Last_Name` | Owner / annuitant identity |
| `1309_Party_ID` | Owner partial ID (last 4 SSN) |

In [0]:
# Load the flattened CSV
pov_csv_path = pov_summary["csv_path"]
pov_df = pd.read_csv(pov_csv_path, dtype=str)

print(f"Shape: {pov_df.shape[0]} contracts x {pov_df.shape[1]} columns\n")

# Show key contract fields
key_cols = [c for c in pov_df.columns if any(k in c for k in [
    "Contract_Number", "CUSIP", "Contract_Status", "Distributors_Account",
    "Product_Type", "Contract_State",
    "Agent_First", "Agent_Last", "National_Producer", "Agent_Tax_ID",
    "Party_First", "Party_Last", "Party_Role", "Party_ID",
    "Party_Date_of_Birth", "Party_Gender",
])]

print("=== Key fields available for XML mapping ===")
for col in key_cols:
    vals = pov_df[col].dropna().unique()
    sample = vals[0] if len(vals) > 0 else "(empty)"
    print(f"  {col:<55s}  {sample}")

print(f"\n=== All {pov_df.shape[1]} columns ===")
for i, col in enumerate(pov_df.columns, 1):
    print(f"  {i:>3d}. {col}")

### Step 3 — Map POV Fields to XML Schema
Build the 21208 Withdrawal-Quote input CSV by:
1. **Extracting** contract/agent/owner data from the flattened POV output
2. **Filling** static protocol fields (SOAP wrapper, tc codes, relation IDs)
3. **Adding** withdrawal-specific parameters (arrangement type, tax withholding, modal amount)

The mapping dictionary `POV_TO_XML` defines which POV column feeds each XML field.
Fields not in the POV data are set to configurable defaults.

In [0]:
import re as _re_mod
import uuid
from datetime import datetime, timezone
from config.pov_record_layouts import (
    FORMAT_EXTENDED, EXTENDED_LINE_WIDTH, STANDARD_LINE_WIDTH,
)

# ── Configurable defaults (adjust per carrier / transaction) ─────────
CARRIER_CODE        = "4500"
DTCC_MEMBER_CODE    = "4602"     # Carrier DTCC member code
DISTR_MEMBER_CODE   = "0015"     # Distributor DTCC member code
ARR_MODE_TC         = "9"        # Single Pay
ARR_MODE_TEXT       = "Single Pay"
ARR_TYPE_TC         = "1004900058"
ARR_TYPE_TEXT       = "OneTimeWithdrawal"
ARR_SUBTYPE_TC      = "4"        # Specified Amount
ARR_SUBTYPE_TEXT    = "Specified Amount"
MODAL_AMT           = "2000.00"  # Default withdrawal amount
SRC_XFER_TC         = "6"
SRC_XFER_TEXT       = "ProrataAllFunds"
TAX_FED_TYPE_TC     = "1"        # Default
TAX_STATE_TYPE_TC   = "1"        # Default

# ── Distributor filter (from widget) ───────────────────────────────
DISTRIBUTOR_FILTER  = dbutils.widgets.get("distributor_filter")

# ── Constants ──────────────────────────────────────────────────────
_CONTRACT_SUFFIX_CODES = {"MV", "XE", "TR"}
_OWNER_ROLES           = {"OK", "HA", "JV", "HC"}
_ANNUITANT_ROLES       = {"G2", "G5"}
_RIDER_SF_TYPES        = {"204", "336", "215"}
_NIPR_REGEX            = _re_mod.compile(r"N(\d{5,10})")
_SUFFIX_REGEX          = _re_mod.compile(r"^(.+?)\s+(MV|TR|XE)\s*$")

POV_TO_XML = {
    "Contract_Number":              "PolNumber",
    "1301_CUSIP_Number":            "CusipNum",
    "1301_Distributors_Account_ID": "DistributorClientAcctNum",
}

# ── Helper functions ───────────────────────────────────────────────

def _strip_contract_suffix(contract_num: str) -> str:
    """Remove trailing 2-char product suffixes (MV, TR, XE) from a contract number."""
    if not contract_num or len(contract_num) < 3:
        return contract_num
    suffix = contract_num[-2:].upper()
    if suffix in _CONTRACT_SUFFIX_CODES:
        return contract_num[:-2].rstrip()
    return contract_num


def _extract_npn_from_agent_last(agent_last_name: str) -> str:
    """
    Extract NPN/NIPR from Agent_Last_Name using regex.

    BHF packs agent routing data into the Agent_Last_Name field.
    The NIPR appears as 'N' followed by 5-10 digits.  The field
    has two formats depending on whether the contract is a base
    or suffix (MV/TR/XE) record:
      - Short (15 chars): 'N1574595   N102'
      - Long  (35 chars): '924116              N1574595   N102'
    The regex handles both layouts reliably.
    """
    if not agent_last_name:
        return ""
    m = _NIPR_REGEX.search(agent_last_name)
    return m.group(1) if m else ""


def _find_col(row, base_col: str) -> str:
    """Resolve a column name that may or may not have an occurrence suffix."""
    if base_col in row.index:
        val = row[base_col]
        return val if pd.notna(val) else ""
    suffixed = f"{base_col}_1"
    if suffixed in row.index:
        val = row[suffixed]
        return val if pd.notna(val) else ""
    return ""


def _extract_composite_name(value: str) -> tuple:
    """Extract (last, first, middle) from a 105-char composite name field."""
    if not value or not value.strip():
        return ("", "", "")
    last  = value[0:35].strip()  if len(value) > 0  else ""
    first = value[35:60].strip() if len(value) > 35 else ""
    mid   = value[60:85].strip() if len(value) > 60 else ""
    return (last, first, mid)


def _find_party_by_role(row, target_roles: set) -> tuple:
    """
    Search 1309 occurrences for a party whose role (shifted Last_Name[0:2])
    matches one of *target_roles*.

    Returns (last, first, middle, party_id).
    Prefers natural-person records (non-empty first name).  Falls back to
    non-natural records (e.g. custodial HA) so the Party_ID is still captured.
    """
    # Pass 1 — prefer natural persons (first name present)
    for occ in range(1, 20):
        for nn_col in [f"1309_Party_Non_Natural_Entity_Name_{occ}",
                       "1309_Party_Non_Natural_Entity_Name"]:
            if nn_col not in row.index:
                continue
            nn_val = row[nn_col]
            if pd.isna(nn_val) or not nn_val.strip():
                continue
            ln_col = nn_col.replace("Non_Natural_Entity_Name", "Last_Name")
            ln_val = row.get(ln_col, "") if ln_col in row.index else ""
            ln_val = ln_val if pd.notna(ln_val) else ""
            role = ln_val[0:2] if len(ln_val) >= 2 else ""
            if role not in target_roles:
                continue
            last, first, mid = _extract_composite_name(nn_val)
            party_id = ln_val[2:22].strip() if len(ln_val) > 2 else ""
            if first:
                return (last, first, mid, party_id)
        if occ == 1 and "1309_Party_Non_Natural_Entity_Name" in row.index:
            break

    # Pass 2 — accept non-natural (e.g. custodial HA) to capture Party_ID
    for occ in range(1, 20):
        for nn_col in [f"1309_Party_Non_Natural_Entity_Name_{occ}",
                       "1309_Party_Non_Natural_Entity_Name"]:
            if nn_col not in row.index:
                continue
            nn_val = row[nn_col]
            if pd.isna(nn_val) or not nn_val.strip():
                continue
            ln_col = nn_col.replace("Non_Natural_Entity_Name", "Last_Name")
            ln_val = row.get(ln_col, "") if ln_col in row.index else ""
            ln_val = ln_val if pd.notna(ln_val) else ""
            role = ln_val[0:2] if len(ln_val) >= 2 else ""
            if role not in target_roles:
                continue
            last, first, mid = _extract_composite_name(nn_val)
            party_id = ln_val[2:22].strip() if len(ln_val) > 2 else ""
            return (last, first, mid, party_id)
        if occ == 1 and "1309_Party_Non_Natural_Entity_Name" in row.index:
            break

    return ("", "", "", "")


def _extract_last4(val):
    """Extract last 4 digits from a possibly masked ID."""
    if pd.isna(val) or not val:
        return ""
    digits = "".join(c for c in str(val) if c.isdigit())
    return digits[-4:] if len(digits) >= 4 else digits


def _build_contra_lookup(pov_path: str) -> dict:
    """
    Read the raw POV file and extract the Contra-Participant ID
    from the 36-byte extension block on each 1301 record.

    Returns a dict mapping contract_number → contra_participant_id.
    Only applicable to extended-format files (336-char lines).
    For standard-format files (300-char lines without extensions),
    returns an empty dict.
    """
    lookup = {}
    with open(pov_path, "r", encoding="utf-8", errors="replace") as fh:
        for raw in fh:
            line = raw.rstrip("\n\r")
            # Only 1301 records in extended format have the extension
            if len(line) != EXTENDED_LINE_WIDTH or line[1:5] != "1301":
                continue
            contract = line[5:35].strip()
            contra   = line[307:311].strip()  # Ext_Contra_Participant at ext offset 7-10
            lookup[contract] = contra
    return lookup


def _contracts_with_qualifier_amt(df, qualifier_code):
    """Return dict mapping Contract_Number → max raw amount for a qualifier."""
    _vq = [c for c in df.columns if "Contract_Value_Qualifier" in c]
    result = {}
    for qcol in _vq:
        acol = qcol.replace("Qualifier", "Amount")
        if acol not in df.columns:
            continue
        mask = df[qcol].fillna("").str.strip() == qualifier_code
        for idx in df[mask].index:
            cn = df.at[idx, "Contract_Number"]
            amt = pd.to_numeric(df.at[idx, acol], errors="coerce")
            if pd.notna(amt) and amt > 0:
                result[cn] = max(result.get(cn, 0), amt)
    return result


def _contracts_with_active_rider(df):
    """Return set of Contract_Numbers with an active rider in 1315 records.

    Rider criteria (all must be true for at least one 1315 occurrence):
      - Service_Feature_Program_Type = 'R'
      - Service_Feature_Type_Code_1  in {204, 336, 215}
      - Service_Feature_Value        > 0
    """
    rider_contracts = set()
    for occ in range(1, 13):
        pt_col  = f"1315_Service_Feature_Program_Type_{occ}"
        tc_col  = f"1315_Service_Feature_Type_Code_1_{occ}"
        val_col = f"1315_Service_Feature_Value_{occ}"
        if pt_col not in df.columns or tc_col not in df.columns:
            continue
        mask = (df[pt_col].fillna("").str.strip() == "R") & \
               (df[tc_col].fillna("").str.strip().isin(_RIDER_SF_TYPES))
        if val_col not in df.columns:
            continue
        for idx in df[mask].index:
            val = pd.to_numeric(df.at[idx, val_col], errors="coerce")
            if pd.notna(val) and val > 0:
                rider_contracts.add(df.at[idx, "Contract_Number"])
    return rider_contracts


def build_xml_rows(pov_df):
    """Convert flattened POV rows into XML-schema-ready dicts."""
    xml_columns = get_column_names()
    now_utc = datetime.now(timezone.utc)
    rows = []

    for _, pov_row in pov_df.iterrows():
        row = {col: "" for col in xml_columns}

        # ── Static SOAP / protocol fields ──
        row["SoapEnvelopeNs"]             = "http://www.w3.org/2003/05/soap-envelope"
        row["SoapBodyPresent"]            = "1"
        row["OperationName"]              = "processValueInquiry21208"
        row["OperationNs"]                = "http://service.iwa.dtcc.com/"
        row["MessageType"]                = "VI21208_Msg"
        row["TXLifeRequest_PrimaryObjectID"] = "Holding_1"
        row["TransRefGUID"]               = str(uuid.uuid4())
        row["TransType_tc"]               = "212"
        row["TransType_text"]             = "Values Inquiry"
        row["TransSubType_tc"]            = "21208"
        row["TransSubType_text"]          = "Withdrawal Quote"
        row["TransExeDate"]               = now_utc.strftime("%Y-%m-%d")
        row["TransExeTime"]               = now_utc.strftime("%H:%M:%S+00:00")
        row["PendingResponseOK_tc"]       = "0"
        row["PendingResponseOK_text"]     = "False"

        # ── Holding / Policy ──
        row["Holding_id"]                 = "Holding_1"
        row["HoldingTypeCode_tc"]         = "2"
        row["HoldingTypeCode_text"]       = "Policy"
        row["Policy_CarrierPartyID"]      = "Party_Carrier"
        row["Policy_CarrierCode"]         = CARRIER_CODE
        row["Annuity_present"]            = "1"

        # ── Arrangement ──
        row["ArrMode_tc"]                 = ARR_MODE_TC
        row["ArrMode_text"]               = ARR_MODE_TEXT
        row["ArrType_tc"]                 = ARR_TYPE_TC
        row["ArrType_text"]               = ARR_TYPE_TEXT
        row["ArrSubType_tc"]              = ARR_SUBTYPE_TC
        row["ArrSubType_text"]            = ARR_SUBTYPE_TEXT
        row["ModalAmt"]                   = MODAL_AMT
        row["SourceTransferAmtType_tc"]   = SRC_XFER_TC
        row["SourceTransferAmtType_text"] = SRC_XFER_TEXT

        # ── Tax Withholding ──
        row["TaxFed_id"]                  = "ID_1"
        row["TaxFed_AppliesToPartyID"]    = "Party_PrimaryOwner"
        row["TaxFed_Place_tc"]            = "1"
        row["TaxFed_Place_text"]          = "Federal"
        row["TaxFed_Type_tc"]             = TAX_FED_TYPE_TC
        row["TaxFed_Type_text"]           = "Default"
        row["TaxState_id"]                = "ID_2"
        row["TaxState_AppliesToPartyID"]  = "Party_PrimaryOwner"
        row["TaxState_Place_tc"]          = "2"
        row["TaxState_Place_text"]        = "State"
        row["TaxState_Type_tc"]           = TAX_STATE_TYPE_TC
        row["TaxState_Type_text"]         = "Default"

        # ── OLifEExtension ──
        row["OLifEExtension_VendorCode"]    = "87726"
        row["OLifEExtension_ExtensionCode"] = "EXT"
        row["AmountQualifier_tc"]           = "2"
        row["AmountQualifier_text"]         = "Gross"

        # ── Map simple POV fields into XML columns ──
        for pov_col, xml_col in POV_TO_XML.items():
            val = _find_col(pov_row, pov_col)
            row[xml_col] = val

        # ── Strip trailing product suffixes (MV/TR/XE) from policy number ──
        row["PolNumber"] = _strip_contract_suffix(row["PolNumber"])

        # ── Agent name (from 1305 composite Non_Natural_Name) ──
        agent_nn = _find_col(pov_row, "1305_Agent_Non_Natural_Name")
        agent_last, agent_first, _ = _extract_composite_name(agent_nn)
        row["Party_Agent_FirstName"] = agent_first
        row["Party_Agent_LastName"]  = agent_last

        # ── Agent NPN (regex from 1305_Agent_Last_Name) ──
        agent_ln = _find_col(pov_row, "1305_Agent_Last_Name")
        row["Party_Agent_NIPRNumber"] = _extract_npn_from_agent_last(agent_ln)

        # ── Agent partial ID ──
        # Agent_Tax_ID is empty in BHF POV data — no SSN available for agent
        row["Party_Agent_IDPart"]              = ""
        row["Party_Agent_PartialIDType_tc"]    = ""
        row["Party_Agent_PartialIDType_text"]  = ""

        # ── Owner name + ID (role-based: OK/HA/JV/HC) ──
        owner_last, owner_first, _, owner_pid = _find_party_by_role(pov_row, _OWNER_ROLES)
        row["Party_PrimaryOwner_FirstName"] = owner_first
        row["Party_PrimaryOwner_LastName"]  = owner_last
        row["Party_PrimaryOwner_IDPart"]    = _extract_last4(owner_pid)
        row["Party_PrimaryOwner_PartialIDType_tc"]   = "1" if owner_pid else ""
        row["Party_PrimaryOwner_PartialIDType_text"] = "SSN" if owner_pid else ""

        # ── Agent party ──
        row["Party_Agent_id"]                 = "Party_Agent"
        row["Party_Agent_PartyTypeCode_tc"]   = "1"
        row["Party_Agent_PartyTypeCode_text"] = "Person"

        # ── Distributor party ──
        row["Party_Distributor_id"]                     = "Party_Distributor"
        row["Party_Distributor_PartyTypeCode_tc"]       = "2"
        row["Party_Distributor_PartyTypeCode_text"]     = "Organization"
        row["Party_Distributor_DTCCMemberCode"]         = DISTR_MEMBER_CODE
        row["Party_Distributor_DTCCAssociatedMemberCode"] = DISTR_MEMBER_CODE

        # ── Carrier party ──
        row["Party_Carrier_id"]                     = "Party_Carrier"
        row["Party_Carrier_PartyTypeCode_tc"]       = "2"
        row["Party_Carrier_PartyTypeCode_text"]     = "Organization"
        row["Party_Carrier_DTCCMemberCode"]         = DTCC_MEMBER_CODE
        row["Party_Carrier_DTCCAssociatedMemberCode"] = DTCC_MEMBER_CODE
        row["Party_Carrier_CarrierCode"]            = CARRIER_CODE

        # ── Owner party ──
        row["Party_PrimaryOwner_id"]                 = "Party_PrimaryOwner"
        row["Party_PrimaryOwner_PartyTypeCode_tc"]   = "1"
        row["Party_PrimaryOwner_PartyTypeCode_text"] = "Person"

        # ── Annuitant (separate role-based lookup: G2/G5) ──
        ann_last, ann_first, _, ann_pid = _find_party_by_role(pov_row, _ANNUITANT_ROLES)
        if not ann_last and not ann_first:
            # Fall back to owner data if no annuitant record found
            ann_last, ann_first, ann_pid = owner_last, owner_first, owner_pid
        row["Party_PrimaryAnnuitant_id"]                 = "Party_PrimaryAnnuitant"
        row["Party_PrimaryAnnuitant_PartyTypeCode_tc"]   = "1"
        row["Party_PrimaryAnnuitant_PartyTypeCode_text"] = "Person"
        row["Party_PrimaryAnnuitant_FirstName"]  = ann_first
        row["Party_PrimaryAnnuitant_LastName"]   = ann_last
        row["Party_PrimaryAnnuitant_IDPart"]     = _extract_last4(ann_pid)
        row["Party_PrimaryAnnuitant_PartialIDType_tc"]   = "1" if ann_pid else ""
        row["Party_PrimaryAnnuitant_PartialIDType_text"] = "SSN" if ann_pid else ""

        # ── Relations ──
        for role, role_tc, role_text, party_id in [
            ("Agent",       "38", "Servicing Agent", "Party_Agent"),
            ("Distributor", "83", "Broker Dealer",  "Party_Distributor"),
            ("Carrier",     "87", "Carrier",        "Party_Carrier"),
            ("Owner",       "8",  "Owner",          "Party_PrimaryOwner"),
            ("Annuitant",   "35", "Annuitant",      "Party_PrimaryAnnuitant"),
        ]:
            pfx = f"Relation_{role}"
            row[f"{pfx}_id"]                        = f"{role}_Relation"
            row[f"{pfx}_OriginatingObjectID"]        = "Holding_1"
            row[f"{pfx}_RelatedObjectID"]            = party_id
            row[f"{pfx}_OriginatingObjectType_tc"]   = "4"
            row[f"{pfx}_OriginatingObjectType_text"] = "Holding"
            row[f"{pfx}_RelatedObjectType_tc"]       = "6"
            row[f"{pfx}_RelatedObjectType_text"]     = "Party"
            row[f"{pfx}_RoleCode_tc"]                = role_tc
            row[f"{pfx}_RoleCode_text"]              = role_text

        # ── Test metadata ──
        row["ExpectedToPass"] = "1"
        row["FailureReason"]  = ""

        rows.append(row)

    return pd.DataFrame(rows)


# ╔══════════════════════════════════════════════════════════════════╗
# ║  STEP 3 PIPELINE: Load → Dedup → Filter → Classify → Map       ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── 3.1 Load flattened POV ─────────────────────────────────────────
pov_csv_path = pov_summary["csv_path"]
pov_df = pd.read_csv(pov_csv_path, dtype=str)
cn_col = "Contract_Number"
pov_df_prededup = pov_df.copy()  # Preserve pre-dedup data for 3c POV source validation

# ── 3.2 Deduplicate by base contract number ────────────────────────
# Contracts with suffixes (MV, TR, XE) are the SAME policy appearing
# multiple times in the POV.  Strip suffixes, then for each base CN
# keep the row with the longest Agent_Last_Name (which contains the
# NIPR at the correct offset).  All other fields are identical
# between base and suffix rows.
_before_dedup = len(pov_df)

# Add helper columns for dedup
pov_df["_base_cn"] = pov_df[cn_col].apply(
    lambda x: _strip_contract_suffix(str(x).strip()) if pd.notna(x) else x
)
_agt_ln_col = next((c for c in pov_df.columns if c.startswith("1305_Agent_Last_Name")), None)
if _agt_ln_col:
    pov_df["_agt_ln_len"] = pov_df[_agt_ln_col].fillna("").str.len()
else:
    pov_df["_agt_ln_len"] = 0

# Sort so longest Agent_Last_Name comes first, then drop duplicates
pov_df = pov_df.sort_values("_agt_ln_len", ascending=False)
pov_df = pov_df.drop_duplicates(subset="_base_cn", keep="first")
pov_df = pov_df.drop(columns=["_agt_ln_len"])

# Overwrite Contract_Number with the stripped base CN
pov_df[cn_col] = pov_df["_base_cn"]
pov_df = pov_df.drop(columns=["_base_cn"]).reset_index(drop=True)

print(f"Dedup: {_before_dedup} → {len(pov_df)} contracts "
      f"({_before_dedup - len(pov_df)} suffix duplicates removed)")

# ── 3.3 Distributor filter (Contra-Participant from extension) ─────
print(f"\nDistributor filter: {DISTRIBUTOR_FILTER}")
contra_lookup_raw = _build_contra_lookup(POV_INPUT_PATH)
# Strip suffixes from lookup keys so suffix-only contracts match after dedup
contra_lookup = {_strip_contract_suffix(k): v for k, v in contra_lookup_raw.items()}
contra_series = pov_df[cn_col].map(contra_lookup)
mask_dist = contra_series == DISTRIBUTOR_FILTER
_before_dist = len(pov_df)
pov_df_filtered = pov_df[mask_dist].copy().reset_index(drop=True)
print(f"  Contracts before: {_before_dist}")
print(f"  Contracts after:  {len(pov_df_filtered)}")

# ── 3.4 Active-status filter (VA/AA only) ──────────────────────────
status_col = next(
    (c for c in pov_df_filtered.columns if "Contract_Status" in c and "1301" in c), None
)
if status_col:
    _before_active = len(pov_df_filtered)
    active_mask = pov_df_filtered[status_col].fillna("").str.strip().isin(["VA", "AA"])
    excluded = pov_df_filtered[~active_mask]
    pov_df_filtered = pov_df_filtered[active_mask].copy().reset_index(drop=True)
    _excluded_statuses = excluded[status_col].fillna("").str.strip().value_counts().to_dict()
    _exc_str = "/".join(f"{k}" for k in sorted(_excluded_statuses.keys()) if k)
    print(f"\n  Active-status filter (VA/AA only):")
    print(f"    Before: {_before_active}  →  After: {len(pov_df_filtered)}")
    print(f"    Excluded {_before_active - len(pov_df_filtered)} non-active contracts ({_exc_str})")

# ── 3.5 Withdrawal type classification ─────────────────────────────
# Priority: RMD > Rider Free > Interest Only > Surrender Free > Partial Withdrawal
rmd_amounts = _contracts_with_qualifier_amt(pov_df_filtered, "RA")
rider_contracts = _contracts_with_active_rider(pov_df_filtered)
sc_amounts = _contracts_with_qualifier_amt(pov_df_filtered, "SC")
tw_amounts = _contracts_with_qualifier_amt(pov_df_filtered, "TW")

import random
random.seed(42)

wd_types = []
wd_value_types = []
wd_values = []

for _, row in pov_df_filtered.iterrows():
    cn = row[cn_col]
    # Total contract value for Amount-based quotes (10% of value)
    total_val = 0.0
    for vc in [c for c in pov_df_filtered.columns if "Contract_Value_Amount" in c]:
        amt = pd.to_numeric(row.get(vc, ""), errors="coerce")
        if pd.notna(amt) and amt > total_val:
            total_val = amt

    if cn in rmd_amounts:
        wtype = "RMD"
        base_amount = rmd_amounts[cn]
    elif cn in rider_contracts:
        wtype = "Rider Free"
        base_amount = total_val
    elif cn in sc_amounts:
        wtype = "Interest Only"
        base_amount = sc_amounts[cn]
    elif cn in tw_amounts:
        wtype = "Surrender Free"
        base_amount = tw_amounts[cn]
    else:
        wtype = "Partial Withdrawal"
        base_amount = total_val

    # 50/50 split: Percent (10%) or Amount (10% of contract value)
    if random.random() < 0.5:
        vtype = "Percent"
        vval = "10.00"
    else:
        vtype = "Amount"
        vval = f"{base_amount * 0.10:.2f}" if base_amount > 0 else "0.00"

    wd_types.append(wtype)
    wd_value_types.append(vtype)
    wd_values.append(vval)

# ── 3.6 Build XML input DataFrame ──────────────────────────────────
xml_input_df = build_xml_rows(pov_df_filtered)
xml_input_df["Withdrawal_Type"]       = wd_types
xml_input_df["Withdrawal_Value_Type"] = wd_value_types
xml_input_df["Withdrawal_Value"]      = wd_values

# ── Summary ────────────────────────────────────────────────────────
print(f"\nWithdrawal type distribution:")
for wt, cnt in xml_input_df["Withdrawal_Type"].value_counts().items():
    pct_cnt = (xml_input_df["Withdrawal_Value_Type"] == "Percent")[
        xml_input_df["Withdrawal_Type"] == wt
    ].sum()
    amt_cnt = cnt - pct_cnt
    print(f"  {wt:<25s}: {cnt:>5d}  (Percent={pct_cnt}, Amount={amt_cnt})")

print(f"\nxml_input_df: {xml_input_df.shape[0]} contracts × {xml_input_df.shape[1]} columns")
print(f"Unique PolNumbers: {xml_input_df['PolNumber'].nunique()} (should equal row count)")
nipr_filled = xml_input_df["Party_Agent_NIPRNumber"].fillna("").str.strip().ne("").sum()
print(f"NIPR populated: {nipr_filled}/{len(xml_input_df)} ({nipr_filled/len(xml_input_df)*100:.1f}%)")


In [0]:
# ── Fix: Extract Owner Party_ID from Last_Name for HA records ──────────
# Root cause (2028-ET38, 18 policies):
#   _find_party_by_role() gates on Non_Natural_Entity_Name being non-empty.
#   HA (custodial/holding account) owner records have:
#     - Last_Name:  "HA" + 20-char Party_ID  (role + SSN)  ✓
#     - Non_Natural_Entity_Name:  EMPTY                     ✗ → skipped
#   Result: Owner SSN never extracted for HA-only owners.
#
# Fix: Post-process xml_input_df to scan Last_Name directly for
#   rows where owner Party_ID is still empty.

_owner_roles_set = {"OK", "HA", "JV", "HC"}
_annui_roles_set = {"G2", "G5"}
_ln_cols = sorted([c for c in pov_df_filtered.columns
                   if c.startswith("1309_Party_Last_Name_")])

def _extract_party_id_from_last_name(pov_row, target_roles):
    """Scan Last_Name columns directly for a matching role and extract party_id."""
    for ln_col in _ln_cols:
        ln_val = pov_row.get(ln_col, "")
        if pd.isna(ln_val) or not str(ln_val).strip():
            continue
        ln_val = str(ln_val)
        role = ln_val[0:2] if len(ln_val) >= 2 else ""
        if role in target_roles:
            party_id = ln_val[2:22].strip() if len(ln_val) > 2 else ""
            if party_id:
                return party_id
    return ""

# ── Fix owner Party_ID ──
_owner_empty = (
    xml_input_df["Party_PrimaryOwner_IDPart"].fillna("").str.strip() == ""
)
_owner_fix_count = 0

for idx in xml_input_df[_owner_empty].index:
    pov_row = pov_df_filtered.iloc[idx]
    pid = _extract_party_id_from_last_name(pov_row, _owner_roles_set)
    if pid:
        last4 = pid[-4:] if len(pid) >= 4 else pid
        xml_input_df.at[idx, "Party_PrimaryOwner_IDPart"] = last4
        xml_input_df.at[idx, "Party_PrimaryOwner_PartialIDType_tc"] = "1"
        xml_input_df.at[idx, "Party_PrimaryOwner_PartialIDType_text"] = "SSN"
        _owner_fix_count += 1

# ── Fix annuitant Party_ID (same pattern may apply) ──
_annui_empty = (
    xml_input_df["Party_PrimaryAnnuitant_IDPart"].fillna("").str.strip() == ""
)
_annui_fix_count = 0

for idx in xml_input_df[_annui_empty].index:
    pov_row = pov_df_filtered.iloc[idx]
    pid = _extract_party_id_from_last_name(pov_row, _annui_roles_set)
    if pid:
        last4 = pid[-4:] if len(pid) >= 4 else pid
        xml_input_df.at[idx, "Party_PrimaryAnnuitant_IDPart"] = last4
        xml_input_df.at[idx, "Party_PrimaryAnnuitant_PartialIDType_tc"] = "1"
        xml_input_df.at[idx, "Party_PrimaryAnnuitant_PartialIDType_text"] = "SSN"
        _annui_fix_count += 1

print(f"ET38 fix — Owner Party_ID extracted from Last_Name:")
print(f"  Owner IDPart:     {_owner_fix_count} rows fixed  "
      f"(of {_owner_empty.sum()} empty)")
print(f"  Annuitant IDPart: {_annui_fix_count} rows fixed  "
      f"(of {_annui_empty.sum()} empty)")

# Verify remaining gaps
_still_empty_owner = (xml_input_df["Party_PrimaryOwner_IDPart"].fillna("").str.strip() == "").sum()
_still_empty_annui = (xml_input_df["Party_PrimaryAnnuitant_IDPart"].fillna("").str.strip() == "").sum()
print(f"\n  Remaining empty Owner IDPart:     {_still_empty_owner}")
print(f"  Remaining empty Annuitant IDPart: {_still_empty_annui}")

In [0]:
# ── Step 3a: Sync XML arrangement fields with withdrawal classification ──
# The build_xml_rows() defaults set ArrSubType tc=4 (Specified Amount)
# and ModalAmt=$2000 for every row.  This cell corrects both based on
# the withdrawal classification from Step 3.
#
# VERIFIED against the working run (run_20260325_005852_0015) XMLs:
#   • ArrSubType tc="4" (Specified Amount)  → dollar amounts
#   • ArrSubType tc="6" (Percentage)        → percent values
#   • ModalAmt = calculated withdrawal value (NOT hardcoded $2000)
#   • AmountQualifier tc="2" (Gross) kept for both sub-types
#   • No type-specific vendor codes — the test harness maps
#     Withdrawal_Type to the carrier API externally.
#
# FIX LOG:
#   • ModalAmt was hardcoded to $2000 for ALL policies — must match
#     the calculated Withdrawal_Value.
#   • ArrSubType must be tc="6" / "Percentage" when
#     Withdrawal_Value_Type == "Percent".

# ── 3a.1  Sync ModalAmt with calculated Withdrawal_Value ──────────
xml_input_df["ModalAmt"] = xml_input_df["Withdrawal_Value"]

# ── 3a.2  Set ArrSubType for Percent rows ───────────────────────
pct_mask = xml_input_df["Withdrawal_Value_Type"] == "Percent"
xml_input_df.loc[pct_mask, "ArrSubType_tc"]   = "6"
xml_input_df.loc[pct_mask, "ArrSubType_text"] = "Percentage"

# Amount rows keep the default tc="4" / "Specified Amount" from build_xml_rows.

# ── Summary ────────────────────────────────────────────────────────
print("Step 3a — Arrangement codes synced with withdrawal classification")
print()
for wtype in xml_input_df["Withdrawal_Type"].unique():
    subset = xml_input_df[xml_input_df["Withdrawal_Type"] == wtype]
    for vtype in ["Amount", "Percent"]:
        vs = subset[subset["Withdrawal_Value_Type"] == vtype]
        if len(vs) == 0:
            continue
        sub_tc = vs["ArrSubType_tc"].iloc[0]
        sub_text = vs["ArrSubType_text"].iloc[0]
        modal_sample = vs["ModalAmt"].head(3).tolist()
        print(f"  {wtype:<25s}  {vtype:<8s}  ArrSubType tc={sub_tc} ({sub_text})")
        print(f"  {'':25s}           ModalAmt samples: {modal_sample}")

### Step 3b — Pre-Generation Validation CSV
Create a timestamped run folder (`run_YYYYMMDD_HHMMSS_<distributor>`) and build a
lightweight validation file with the key fields that will feed the XML generator.
Use this to audit data quality **before** committing to a full XML run.

**Step 3 pre-filters:**
* Non-active contracts excluded: only `VA` (active variable) and `AA` (active non-variable) are submitted.
  `DA` (death/terminated), `SU` (surrendered), `OD` (other/deactivated) are dropped.

**Step 3 classification (current POV data):**
* RMD **ENABLED** — `RA` qualifier in 1302 with value > 0 indicates RMD eligibility.
  `RM` is the calculated RMD dollar amount only, not an enrollment indicator.
* Rider Free **ENABLED** — 1315 Service Feature records with `Program_Type = R` and
  `SF_Type` (Type_Code_1) in {`204`, `336`, `215`} and `Value > 0` indicate an active rider.
* Interest Only **ENABLED** — `SC` (Surrender Charge) qualifier in 1302 with value > 0
  indicates active surrender charges; withdrawal limited to interest only.
* Four columns are inserted before `PolNumber` via POV qualifier analysis:

| Column | Logic |
|--------|-------|
| `Withdrawal_Type` | **RMD** (1302 RA > 0) → **Rider Free** (1315 rider active) → **Interest Only** (1302 SC > 0) → **Surrender Free** (1302 TW > 0) → **Partial Withdrawal** (default) |
| `Withdrawal_Value_Type` | **Percent** or **Amount** (50/50 random split within each type) |
| `Withdrawal_Value` | `10` for Percent; 10% of contract value (SU→AV→AM→CR) for Amount |

| Rider Detection | Source |
|--------|--------|
| `Program_Type` | 1315 `Service_Feature_Program_Type` — must equal `R` |
| `SF_Type` | 1315 `Service_Feature_Type_Code_1` — must be `204`, `336`, or `215` |
| `Value` | 1315 `Service_Feature_Value` — must be > 0 |

| Validation Column | Source |
|--------|--------|
| `PolNumber` | 1301 Contract Number (whitespace-stripped, MV/TR/XE suffix removed) |
| `CusipNum` | 1301 CUSIP |
| `NIPRNumber` | 1305 Agent NPN (from shifted layout) |
| `AgtFirstName / AgtLastName` | 1305 composite name |
| `AgtSSN` | 1305 Agent Tax ID (empty in BHF POV) |
| `OwnFirstName / OwnLastName` | 1309 composite name |
| `OwnSSN` | 1309 Party ID last 4 (shifted layout) |
| `AnnSSN` | Same as owner (annuitant = owner) |
| `Rguid` | Generated TransRefGUID |
| `Federal/State_TaxWithholdingType` | Tax withholding defaults |
| `State` | 1301 Contract State |
| `Result` | Blank — populated during validation |

In [0]:
from datetime import datetime, timezone as _tz_val

def build_validation_df(xml_df, pov_df_src):
    """
    Build a pre-generation validation DataFrame from the XML-mapped data
    and supplemental POV fields.

    Parameters
    ----------
    xml_df : pd.DataFrame
        Output of build_xml_rows() — the full XML-schema-ready DataFrame,
        with Withdrawal_Type / Withdrawal_Value_Type / Withdrawal_Value
        columns inserted by the Step 3 classification block.
    pov_df_src : pd.DataFrame
        The filtered POV DataFrame (same row order as xml_df).

    Returns
    -------
    pd.DataFrame with columns:
        Withdrawal_Type, Withdrawal_Value_Type, Withdrawal_Value,
        PolNumber, CusipNum, NIPRNumber, AgtFirstName, AgtLastName, AgtSSN,
        OwnFirstName, OwnLastName, OwnSSN, AnnSSN, Rguid,
        Federal_TaxWithholdingType, State_TaxWithholdingType, State
    """
    # ── Map XML columns to validation columns ──
    val = pd.DataFrame()

    # ── Withdrawal classification (from Step 3) ──
    val["Withdrawal_Type"]       = xml_df["Withdrawal_Type"]
    val["Withdrawal_Value_Type"] = xml_df["Withdrawal_Value_Type"]
    val["Withdrawal_Value"]      = xml_df["Withdrawal_Value"]

    val["PolNumber"]     = xml_df["PolNumber"].str.replace(r"\s+", "", regex=True)
    val["CusipNum"]      = xml_df["CusipNum"]
    val["NIPRNumber"]    = xml_df["Party_Agent_NIPRNumber"]
    val["AgtFirstName"]  = xml_df["Party_Agent_FirstName"]
    val["AgtLastName"]   = xml_df["Party_Agent_LastName"]
    val["AgtSSN"]        = xml_df["Party_Agent_IDPart"]
    val["OwnFirstName"]  = xml_df["Party_PrimaryOwner_FirstName"]
    val["OwnLastName"]   = xml_df["Party_PrimaryOwner_LastName"]
    val["OwnSSN"]        = xml_df["Party_PrimaryOwner_IDPart"]
    val["AnnSSN"]        = xml_df["Party_PrimaryAnnuitant_IDPart"]
    val["Rguid"]         = xml_df["TransRefGUID"]
    val["Federal_TaxWithholdingType"] = xml_df["TaxFed_Type_text"]
    val["State_TaxWithholdingType"]   = xml_df["TaxState_Type_text"]

    # ── State: pull from POV 1301_Contract_State ──
    state_col = next(
        (c for c in pov_df_src.columns if "Contract_State" in c), None
    )
    if state_col:
        val["State"] = pov_df_src[state_col].values
    else:
        val["State"] = ""

    return val


# ── Create timestamped run folder ──
_ts = datetime.now(_tz_val.utc).strftime("%Y%m%d_%H%M%S")
_tag = DISTRIBUTOR_FILTER if DISTRIBUTOR_FILTER and DISTRIBUTOR_FILTER != "ALL" else "ALL"
RUN_DIR = os.path.join(BASE_DIR, "runs", f"run_{_ts}_{_tag}")
os.makedirs(RUN_DIR, exist_ok=True)

# ── Save XML input CSV into the run folder ──
# This is the full schema CSV that Step 4 (run_pipeline) reads.
XML_CSV_PATH = os.path.join(RUN_DIR, f"xml_input_{_ts}_{_tag}.csv")
xml_input_df.fillna("").to_csv(XML_CSV_PATH, index=False)

# ── Build and save validation CSV into the run folder ──
validation_df = build_validation_df(xml_input_df, pov_df_filtered)

VALIDATION_CSV_PATH = os.path.join(RUN_DIR, f"pre_generation_validation_{_ts}.csv")
validation_df.to_csv(VALIDATION_CSV_PATH, index=False)

print(f"Run folder:      {RUN_DIR}")
print(f"XML input CSV:   {XML_CSV_PATH}")
print(f"Validation CSV:  {VALIDATION_CSV_PATH}")
print(f"Rows: {len(validation_df)}  |  Columns: {list(validation_df.columns)}")
print(f"\n── Coverage ──")
for col in validation_df.columns:
    filled = (validation_df[col].astype(str).str.strip().replace("", pd.NA).notna()).sum()
    print(f"  {col:<30s}  {filled:>5d} / {len(validation_df)}")

print(f"\n── Sample (first 5 rows) ──")
display(validation_df.head(5))

### Step 3c — Pre-Generation Failures Analysis
Audit every contract for **missing mandatory fields** that would cause carrier rejection.
Enriches the Step 3b validation CSV with `Failure_Reasons` and `Predicted_Error_Codes`,
then **overwrites** the same file so the full picture is in one place.

| Check | Mandatory Field | DTCC Error Code |
|-------|----------------|------------------|
| NIPR/NPN | `NIPRNumber` | 2108-C075 |
| Owner First Name | `OwnFirstName` | 2108-C075 |
| Owner Last Name | `OwnLastName` | 2108-C075 |
| Owner SSN (last 4) | `OwnSSN` | 2108-C075 / 2028-ET38 |
| Annuitant SSN (last 4) | `AnnSSN` | 2108-C075 |
| Agent SSN (last 4) | `AgtSSN` | 100-BA11 (blanket warning) |
| Withdrawal Value > 0 | `Withdrawal_Value` | 1004900735-X173 |
| Owner+SSN both missing | `OwnFirstName` + `OwnSSN` | 2028-ET38 (identity gap) |
| Surrender Free arrangement | `Withdrawal_Type` | 100-F058 / 100-A687 (now fixed in Step 3a) |

In [0]:
# ── Step 3c: Flag contracts with missing mandatory fields ─────────
# Enriches the Step 3b validation CSV with failure analysis columns
# and overwrites the same file so the full picture is in one place.
# Run this BEFORE Step 4 to audit data quality.

# ── Mandatory field rules ──
# Each tuple: (validation_col, human_label, likely_carrier_error)
# NOTE: AgtSSN excluded from per-contract checks because the POV file
#       does not supply agent SSN for ANY contract.  This is a blanket
#       data-source limitation, not a per-contract issue.
_MANDATORY_CHECKS = [
    ("NIPRNumber",    "Agent NPN/NIPR missing",          "2108-C075"),
    ("OwnFirstName",  "Owner first name missing",        "2108-C075"),
    ("OwnLastName",   "Owner last name missing",         "2108-C075"),
    ("OwnSSN",        "Owner SSN (last 4) missing",      "2108-C075"),
    ("AnnSSN",        "Annuitant SSN (last 4) missing",  "2108-C075"),
]

def _is_empty(val):
    """True if value is NaN, empty string, whitespace-only, or literal 'nan'."""
    if pd.isna(val):
        return True
    s = str(val).strip()
    return s == "" or s.lower() == "nan"


def enrich_validation_df(val_df):
    """
    Scan every row for missing mandatory fields and zero withdrawal
    values.  Adds Failure_Reasons and Predicted_Error_Codes columns.

    Returns the enriched DataFrame (all rows, not just failures).
    """
    reasons_list = []
    codes_list = []

    for _, row in val_df.iterrows():
        reasons = []
        codes = set()

        # ── mandatory field checks ──
        for col, label, code in _MANDATORY_CHECKS:
            if _is_empty(row.get(col, "")):
                reasons.append(label)
                codes.add(code)

        # ── withdrawal value > 0 check ──
        try:
            wv = float(row.get("Withdrawal_Value", 0))
        except (ValueError, TypeError):
            wv = 0.0
        if wv <= 0:
            reasons.append("Withdrawal value is 0 or missing")
            codes.add("1004900735-X173")

        reasons_list.append("; ".join(reasons) if reasons else "")
        codes_list.append("; ".join(sorted(codes)) if codes else "")

    val_df = val_df.copy()
    val_df["Failure_Reasons"]       = reasons_list
    val_df["Predicted_Error_Codes"] = codes_list
    return val_df


# ── Run the audit and overwrite the Step 3b CSV ──
validation_df = enrich_validation_df(validation_df)
validation_df.to_csv(VALIDATION_CSV_PATH, index=False)

# ── Summary ──
_pass_count = (validation_df["Predicted_Error_Codes"] == "").sum()
_fail_count = len(validation_df) - _pass_count

print(f"Validation CSV:  {VALIDATION_CSV_PATH}  (enriched with failure analysis)")
print(f"")
print(f"Total contracts:  {len(validation_df)}")
print(f"  PASS:           {_pass_count}")
print(f"  FAIL:           {_fail_count}")
print(f"")

# ── Blanket warnings (POV data-source gaps) ──
agt_ssn_empty = (validation_df["AgtSSN"].fillna("").str.strip() == "").sum()
if agt_ssn_empty == len(validation_df):
    print(f"⚠ BLANKET WARNING: AgtSSN is empty for ALL {agt_ssn_empty} contracts")
    print(f"  (POV file does not supply agent SSN — potential 100-BA11 risk)\n")

# ── Breakdown by failure reason ──
failures_df = validation_df[validation_df["Predicted_Error_Codes"] != ""]
if len(failures_df) > 0:
    print("── Failure Reason Breakdown ──")
    reason_counts = {}
    for reasons_str in failures_df["Failure_Reasons"]:
        for r in reasons_str.split("; "):
            reason_counts[r] = reason_counts.get(r, 0) + 1
    for reason, count in sorted(reason_counts.items(), key=lambda x: -x[1]):
        print(f"  {reason:<40s} {count:>5d}")

    # ── Breakdown by predicted error code ──
    print(f"\n── Predicted Carrier Error Codes ──")
    code_counts = {}
    for codes_str in failures_df["Predicted_Error_Codes"]:
        for c in codes_str.split("; "):
            code_counts[c] = code_counts.get(c, 0) + 1
    for code, count in sorted(code_counts.items(), key=lambda x: -x[1]):
        print(f"  {code:<25s} {count:>5d}")

    # ── Contracts with multiple issues ──
    multi = failures_df[failures_df["Failure_Reasons"].str.contains(";")]
    print(f"\n  Contracts with multiple issues: {len(multi)}")

    print(f"\n── Sample failures (first 10) ──")
    display(failures_df[["PolNumber", "Withdrawal_Type", "NIPRNumber", "OwnFirstName",
                          "OwnSSN", "AnnSSN", "Failure_Reasons", "Predicted_Error_Codes"]].head(10))
else:
    print("✓ All contracts passed mandatory field checks.")

# ══════════════════════════════════════════════════════════════════
# POV SOURCE VALIDATION
# ══════════════════════════════════════════════════════════════════
# Cross-check each missing field against the raw (pre-dedup) POV to
# determine if data is genuinely absent from the POV source vs lost
# during dedup row selection.

if len(failures_df) > 0 and 'pov_df_prededup' in dir():
    print("\n" + "="*60)
    print("POV SOURCE VALIDATION")
    print("="*60)
    print("Cross-checking each missing field against ALL raw POV rows")
    print("(pre-dedup) to confirm: genuinely absent vs. dedup artifact.\n")

    # Build base_cn → list of row indices in pov_df_prededup
    _raw = pov_df_prededup.copy()
    _raw["_base_cn"] = _raw["Contract_Number"].apply(
        lambda x: _strip_contract_suffix(str(x).strip()) if pd.notna(x) else x
    )

    from collections import defaultdict
    _cn_to_rows = defaultdict(list)
    for _ridx in _raw.index:
        _cn_to_rows[_raw.at[_ridx, "_base_cn"]].append(_ridx)

    _agt_ln_col_raw = next(
        (c for c in _raw.columns if c.startswith("1305_Agent_Last_Name")), None
    )

    _field_checks = [
        ("Agent NPN/NIPR missing",         "nipr"),
        ("Owner first name missing",       "own_first"),
        ("Owner last name missing",        "own_last"),
        ("Owner SSN (last 4) missing",     "own_ssn"),
        ("Annuitant SSN (last 4) missing", "ann_ssn"),
    ]

    _summary_rows = []

    for _reason_label, _check_type in _field_checks:
        _failing_pols = failures_df[
            failures_df["Failure_Reasons"].str.contains(_reason_label, na=False, regex=False)
        ]["PolNumber"].tolist()

        if not _failing_pols:
            continue

        _genuinely_absent = 0
        _had_in_other_row = 0

        for _pol in _failing_pols:
            _raw_indices = _cn_to_rows.get(_pol.strip(), [])
            _found_any = False

            for _ri in _raw_indices:
                _raw_row = _raw.loc[_ri]

                if _check_type == "nipr":
                    _aln = _raw_row.get(_agt_ln_col_raw, "") if _agt_ln_col_raw else ""
                    _aln = _aln if pd.notna(_aln) else ""
                    if _NIPR_REGEX.search(_aln):
                        _found_any = True
                        break

                elif _check_type in ("own_first", "own_last", "own_ssn"):
                    _ol, _of, _, _op = _find_party_by_role(_raw_row, _OWNER_ROLES)
                    if _check_type == "own_first" and _of:
                        _found_any = True; break
                    elif _check_type == "own_last" and _ol:
                        _found_any = True; break
                    elif _check_type == "own_ssn" and _extract_last4(_op):
                        _found_any = True; break

                elif _check_type == "ann_ssn":
                    _al, _af, _, _ap = _find_party_by_role(_raw_row, _ANNUITANT_ROLES)
                    if _extract_last4(_ap):
                        _found_any = True; break

            if _found_any:
                _had_in_other_row += 1
            else:
                _genuinely_absent += 1

        _total = len(_failing_pols)
        _summary_rows.append({
            "Missing Field": _reason_label.replace(" missing", ""),
            "Total Missing": _total,
            "Genuinely Absent (POV)": _genuinely_absent,
            "Dedup Artifact": _had_in_other_row,
        })

        print(f"  {_reason_label}:")
        print(f"    Total missing:              {_total:>5d}")
        print(f"    Genuinely absent from POV:  {_genuinely_absent:>5d}")
        print(f"    Was in alternate POV row:    {_had_in_other_row:>5d}")
        print()

    if _summary_rows:
        _pov_val_df = pd.DataFrame(_summary_rows)
        print("── POV Source Validation Summary ──")
        display(_pov_val_df)


### Step 4 — Generate ACORD 21208 XMLs
Run the full XML pipeline: DDL → CSV load → XML generation → validation → scorecard → sort.

Reuses the run folder created in **Step 3b** so the validation CSV and XML output
live together in the same timestamped directory.

In [0]:
import modules.table_manager as _tm
_tm.load_csv = _load_csv_serverless  # patch for serverless (lazy import inside run_pipeline)

# RUN_DIR already created in Step 3b with timestamp + distributor tag
print(f"Run folder:  {RUN_DIR}")
print(f"Contracts:   {len(xml_input_df)}")
print(f"Distributor: {_tag}\n")

summary = run_pipeline(
    base_dir=RUN_DIR,
    csv_source=XML_CSV_PATH,
    table_name="wd_quote_21208",
)

print(f"\n{'='*60}")
print(f"Run folder       : {RUN_DIR}")
print(f"Validation CSV   : {VALIDATION_CSV_PATH}")
print(f"XMLs generated   : {summary['xml_count']}")
print(f"Passed validation: {summary['passed']}")
print(f"Failed validation: {summary['failed']}")
print(f"Scorecard        : {summary['scorecard_path']}")
print(f"Success dir      : {summary['success_dir']}")
print(f"Unsuccessful dir : {summary['unsuccessful_dir']}")

### Step 5 — Evaluate Carrier Responses
Load the DTCC results XLSX and cross-reference with our pre-generation validation CSV
to produce a comprehensive error analysis.

| Error Category | Count | Code | Root Cause | Status |
|---------------|-------|------|------------|--------|
| **Distributor Mismatch** | 228 | 2109-None | POV contra-participant stale/incorrect | Data-side; exclude or re-map |
| **Surrender Free Rejection** | 34 | 100-F058 / 100-A687 | Carrier-side validation for Surrender Free contracts | Investigate carrier field requirements |
| **Rider Amt Zero** | 30 | 1004900735-X173 | Carrier rider value depleted/expired | Flag in pre-gen; skip or re-classify |
| **Owner Data Missing** | 18 | 2028-ET38 | Owner party extraction gap (missing name + SSN) | Widen `_OWNER_ROLES` or investigate POV |
| **Terminated Contract** | 3 | 1004903053 | Active in POV, terminated at carrier | POV data lag; carrier-side |
| **System/Transient** | 13 | 300 / 3013 | DTCC system unavailable / internal error | Retry |

**Step 3a fixes** (verified against working run XMLs):
* `ModalAmt` synced to calculated `Withdrawal_Value` (was hardcoded `$2000`)
* `ArrSubType` tc=**6** / "Percentage" for percent-based quotes (was always tc=4)
* `XML_CSV_PATH` now saved into run folder (was missing from notebook flow)

In [0]:
import json as _json_mod
import pandas as pd

# ╔══════════════════════════════════════════════════════════════╗
# ║  Path to carrier response XLSX — update after each run      ║
# ╚══════════════════════════════════════════════════════════════╝
RESULTS_XLSX = os.path.join(
    PROJECT_ROOT,
    "Copy of WDQuote Results_20260325_1810 Policies.xlsx",
)

results_df = pd.read_excel(RESULTS_XLSX, dtype=str)
print(f"Results file: {RESULTS_XLSX}")
print(f"Rows: {len(results_df)}  |  Columns: {list(results_df.columns)}")

# ── Parse error codes ─────────────────────────────────────────────
def _parse_error(desc):
    if pd.isna(desc):
        return []
    try:
        items = _json_mod.loads(desc)
        return [(i.get("ResultInfoCode", ""), i.get("ResultInfoDesc", "")) for i in items]
    except (ValueError, TypeError):
        return [(str(desc)[:40], str(desc))]

results_df["_errors"] = results_df["Error_Descriptions"].apply(_parse_error)
results_df["_error_codes"] = results_df["_errors"].apply(
    lambda es: "; ".join(c for c, _ in es) if es else ""
)

# ── High-level summary ────────────────────────────────────────────
total   = len(results_df)
success = (results_df["Status"] == "Success").sum()
failure = total - success
print(f"\n{'='*60}")
print(f"CARRIER RESPONSE SUMMARY")
print(f"{'='*60}")
print(f"Total requests:    {total:>6d}")
print(f"Success:           {success:>6d}  ({success/total*100:.1f}%)")
print(f"Failure:           {failure:>6d}  ({failure/total*100:.1f}%)")
print(f"Result code 2 (OK w/ warnings): {(results_df['Result_code'] == '2').sum()}")

# ── Error code breakdown ──────────────────────────────────────────
fail_df = results_df[results_df["Status"] == "Failure"].copy()
print(f"\n{'='*60}")
print(f"FAILURE BREAKDOWN BY ERROR CODE")
print(f"{'='*60}")

# Build a tidy error summary
error_rows = []
for _, row in fail_df.iterrows():
    for code, desc in row["_errors"]:
        error_rows.append({
            "Error_Code": code.strip(),
            "Description": desc[:80],
            "Withdrawal_Type": row["Withdrawal_Type"],
            "Policy_Number": row["Policy_Number"],
        })
    if not row["_errors"]:
        error_rows.append({
            "Error_Code": row.get("_error_codes", "UNKNOWN"),
            "Description": str(row["Error_Descriptions"])[:80],
            "Withdrawal_Type": row["Withdrawal_Type"],
            "Policy_Number": row["Policy_Number"],
        })

err_tidy = pd.DataFrame(error_rows)
code_summary = err_tidy.groupby("Error_Code").agg(
    Count=("Policy_Number", "nunique"),
    Description=("Description", "first"),
    Top_WD_Type=("Withdrawal_Type", lambda x: x.value_counts().index[0]),
).sort_values("Count", ascending=False)

for code, r in code_summary.iterrows():
    tag = ""
    if "2109" in code:         tag = "[DATA] POV contra-participant stale"
    elif "F058" in code:       tag = "[CARRIER] Surrender Free field validation"
    elif "A687" in code:       tag = "[CARRIER] Surrender Free field validation"
    elif "X173" in code:       tag = "[DATA] Carrier rider value = 0"
    elif "ET38" in code:       tag = "[CODE] Owner party extraction gap"
    elif "1004903053" in code: tag = "[DATA] POV status lag"
    elif "300" in code:        tag = "[TRANSIENT] System unavailable"
    elif "3013" in code:       tag = "[TRANSIENT] Internal error"
    elif "2027" in code:       tag = "[DATA] Contract restricted"
    elif "BA11" in code:       tag = "[KNOWN] Agent SSN unavailable in POV"
    elif "WDS3" in code:       tag = "[INFO] Disclosure warning"
    elif "A452" in code:       tag = "[INVESTIGATE] Missing field A452"
    elif "E487" in code:       tag = "[INVESTIGATE] Missing field E487"
    print(f"  {code:<25s}  {r['Count']:>4d}  {r['Top_WD_Type']:<30s}  {tag}")

# ── Success rate by Withdrawal Type ──────────────────────────────
print(f"\n{'='*60}")
print(f"SUCCESS RATE BY WITHDRAWAL TYPE")
print(f"{'='*60}")
for wtype, grp in results_df.groupby("Withdrawal_Type"):
    ok = (grp["Status"] == "Success").sum()
    tot = len(grp)
    rate = ok / tot * 100
    bar = "█" * int(rate / 2) + "░" * (50 - int(rate / 2))
    print(f"  {wtype:<35s}  {ok:>5d}/{tot:<5d}  {rate:5.1f}%  {bar}")

# ── Categorized failure summary ─────────────────────────────────
data_issues  = fail_df[fail_df["_error_codes"].str.contains("2109|1004903053|2027|X173", na=False)]
carrier_val  = fail_df[fail_df["_error_codes"].str.contains("F058|A687|ET38|A452|E487|BA11", na=False)]
transient    = fail_df[fail_df["_error_codes"].str.contains("300-System|3013", na=False)]
counted      = set(data_issues.index) | set(carrier_val.index) | set(transient.index)
remaining    = len(fail_df) - len(counted)

print(f"\n{'='*60}")
print(f"CATEGORIZED FAILURE SUMMARY")
print(f"{'='*60}")
print(f"  Data / carrier mismatch:         {len(data_issues):>4d}  (POV data stale vs carrier)")
print(f"  Carrier field validation:        {len(carrier_val):>4d}  (missing/invalid fields)")
print(f"  Transient (retry):               {len(transient):>4d}")
print(f"  Other:                           {remaining:>4d}")
print(f"")
print(f"Step 3a fixes (ModalAmt + ArrSubType) resolve the code-side")
print(f"regression. Remaining failures are carrier/data issues.")

# ── Cross-reference with validation CSV (if available) ──────────
if 'VALIDATION_CSV_PATH' in dir() and os.path.isfile(VALIDATION_CSV_PATH):
    _val = pd.read_csv(VALIDATION_CSV_PATH, dtype=str)
    _val["_pol_clean"] = _val["PolNumber"].str.strip()
    results_df["_pol_clean"] = results_df["Policy_Number"].str.strip().str.lstrip("0")
    _val["_pol_clean_nolz"] = _val["_pol_clean"].str.lstrip("0")
    _merged = results_df.merge(
        _val, left_on="_pol_clean", right_on="_pol_clean_nolz", how="left",
        suffixes=("_result", "_val")
    )
    _matched = _merged["PolNumber"].notna().sum()
    print(f"\n  Validation CSV cross-ref: {_matched}/{len(results_df)} matched")
    if "Predicted_Error_Codes" in _val.columns:
        _predicted_fail = (_val["Predicted_Error_Codes"].fillna("") != "").sum()
        print(f"  Pre-gen predicted failures: {_predicted_fail}")
        print(f"  Actual failures:            {failure}")